# 06 · RAG 检索增强（给模型外挂真实法条，治法条幻觉）

> **项目的改进增量。** 05 发现：SFT/DPO 解决了"形式"，但法条事实幻觉仍在（把重婚罪说成第 313 条）。
> **RAG 思路**：回答前先**检索真实法条**，拼进 prompt，让模型**照着真法条答**。
> 关键：模型参数**完全不变**（不训练），只是推理时多给一段"开卷资料"（in-context）。

## 这是学习级 baseline（不是生产最佳实践）
| 环节 | 本 demo | 生产级会升级成 |
|---|---|---|
| embedding | bge-small-zh（轻量） | bge-large / 法律领域微调 |
| 检索 | numpy 余弦（数据小） | 向量数据库 + 混合检索(向量+BM25) + rerank |
| 生成 | 复用 04 的 dpo_merged | 同上 + 引用标注/防注入 |
| 评测 | 人工看 | RAGAS（忠实度/引用正确性） |

> ⚠️ `data/clauses.jsonl` 是凭记忆起草的骨架，**务必去 flk.npc.gov.cn 逐条核对原文**——这正是 RAG 的精神：别信模型记忆，以权威库为准。

In [ ]:
import os, json
import torch
import torch.nn.functional as F   # F.normalize 做向量归一化要用

# 走镜像 + 修证书（和前面 notebook 一致）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import certifi; os.environ["SSL_CERT_FILE"] = certifi.where()

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA = os.path.join(ROOT, "data")
OUT  = os.path.join(ROOT, "outputs")
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

BGE_PATH   = os.path.join(ROOT, "models", "bge-small-zh-v1.5")  # 检索用的 embedding 模型
DPO_MERGED = os.path.join(OUT, "dpo_merged")                    # 04 训好的最终模型，当生成器
print("DEVICE:", DEVICE)

In [ ]:
# ---- 加载 embedding 模型 bge，定义"文字→向量"的 embed 函数 ----
from transformers import AutoTokenizer, AutoModel

bge_tok = AutoTokenizer.from_pretrained(BGE_PATH)
bge = AutoModel.from_pretrained(BGE_PATH).to(DEVICE).eval()   # eval()=推理模式(关 dropout)

def embed(texts):
    # texts: 字符串列表 → 返回 (条数, 向量维度) 的张量
    enc = bge_tok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
    with torch.no_grad():                       # 只做推理，不算梯度，省内存
        out = bge(**enc)
    vec = out.last_hidden_state[:, 0]           # 取每条的第 0 个 token(CLS)当句向量(bge 推荐用法)
    vec = F.normalize(vec, p=2, dim=1)          # L2 归一化：归一化后两向量点积 = 余弦相似度
    return vec

print("bge 加载完成，向量维度:", embed(["测试"]).shape[1])

In [ ]:
# ---- 读法条库，把每条法条编码成向量（建"索引"）----
clauses = [json.loads(l) for l in open(os.path.join(DATA, "clauses.jsonl"), encoding="utf-8") if l.strip()]
print("法条数:", len(clauses))

# 把每条法条的正文编码成向量，堆成一个矩阵 (法条数, 向量维度)
clause_vecs = embed([c["text"] for c in clauses])
print("法条向量矩阵:", clause_vecs.shape)

In [ ]:
# ---- 检索函数：给一个问题，找出最相关的 k 条法条 ----
def retrieve(query, k=3):
    qv = embed([query])                 # 问题编码成向量 (1, dim)
    sims = (qv @ clause_vecs.T)[0]      # 问题向量 × 所有法条向量 → 每条的相似度 (法条数,)
    topk = sims.topk(k)                 # 取相似度最高的 k 条
    return [(clauses[i], float(s)) for i, s in zip(topk.indices.tolist(), topk.values.tolist())]

# 测试检索：问重婚罪，看能不能捞出刑法第258条
for c, score in retrieve("什么是重婚罪", k=3):
    print(f"{score:.3f}  {c['law']}{c['article']}")

In [ ]:
# ---- 加载生成模型(04 的 dpo_merged)，复用 03/05 的 generate ----
from transformers import AutoModelForCausalLM

gen_tok = AutoTokenizer.from_pretrained(DPO_MERGED)
gen_model = AutoModelForCausalLM.from_pretrained(DPO_MERGED, torch_dtype=torch.float32).to(DEVICE)

def generate(system, user, max_new_tokens=256):
    msgs = [{"role":"system","content":system}, {"role":"user","content":user}]
    text = gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = gen_tok(text, return_tensors="pt").to(gen_model.device)
    out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("生成模型就绪")

In [ ]:
# ---- 两种回答方式：无 RAG vs 有 RAG ----
SYSTEM = "你是专业的中文法律咨询助手。请针对用户的法律问题，给出准确、有依据、条理清晰的解答；涉及具体法条时尽量说明，不要编造不存在的法条或事实。"

def answer_no_rag(q):
    return generate(SYSTEM, q)                       # 直接问，靠模型记忆(会编法条)

def answer_rag(q, k=3):
    hits = retrieve(q, k)                            # 1) 先检索真实法条
    ctx = "\n".join(f"{c['law']}{c['article']}：{c['text']}" for c, _ in hits)
    # 2) 把法条拼进 prompt，明确要求"只依据这些法条、不要编造"
    user = (f"请只依据下面提供的法条回答问题，不得编造法条编号或内容，并指出你引用了哪一条。\n\n"
            f"【参考法条】\n{ctx}\n\n【问题】{q}")
    return hits, generate(SYSTEM, user)              # 3) 喂给模型(参数没变，只是输入多了法条)

In [ ]:
# ---- demo：并排对比 无RAG / 有RAG ----
questions = ["啥样算重婚罪的具体行为？", "婚前买房离婚属于共同财产吗？"]

for q in questions:
    print("="*70)
    print("问题:", q)
    print("-"*70)
    print("【无 RAG】", answer_no_rag(q)[:220])
    print("-"*70)
    hits, ans = answer_rag(q)
    print("检索到:", [f"{c['law']}{c['article']}" for c, _ in hits])
    print("【有 RAG】", ans[:300])
    print()

## 想一想（面试会问）
1. RAG 为什么不用重新训练模型就能"治幻觉"？（提示：in-context vs in-weight）
2. 如果**检索错了法条**（召回了不相关的），RAG 的回答会怎样？说明了什么？
3. 这个 demo 用 numpy 暴力检索，数据涨到 10 万条法条时会有什么问题？该换成什么？

## 改进方向（baseline → 生产）
- 检索：向量数据库(Milvus/Qdrant) + **混合检索(向量+BM25) + rerank**（法条号、专名靠关键词更准）
- embedding：换 bge-large 或**用法律数据微调**的 embedding
- 生成：引用标注、上下文压缩、防 prompt 注入、**检索不到就拒答**
- 评测：用 **RAGAS** 测 faithfulness（答案是否忠于检索法条）、引用正确性、context recall
- 进阶：retrieval-augmented SFT —— 训练数据构造成 (问题+检索法条)→答案，让模型更会用检索结果